# 15.01 · Decision models vs LLMs: first calls to Jev

**Module 15 · Decision Models with Jev** · Part 2 (Pro) · optional

| | |
|---|---|
| ⏱️ Estimated time | 2–3 h |
| 📋 Prerequisites | Modules 01–02 (models, structured output) and Module 14's ticket router. Read the module README first. |
| 🔑 Keys | optional `TYPESAFE_API_KEY` (paid, no free tier; replay mode without it) · optional `GROQ_API_KEY` for the LLM comparison |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-16 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (no network, free) · **⚖️** Jev: live with `TYPESAFE_API_KEY` (paid, fractions of a cent), otherwise replayed from recordings · **💸** calls Groq (uses your free-tier quota).
> Without keys, ⚖️ cells replay recorded answers, 💸 cells print `⏭️ Skipped`, and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Explain** what a *decision model* (TypeSafe's "System One" model) is, and how it differs from a chat LLM with structured output.
2. **Set up** Jev access: what costs money, what is free, and how replay mode lets you learn without a key.
3. **Ask** the three question types (`Noul`, `Choice`, `Score`) with `TypeSafeClassifier`, and read the typed answers, probabilities and confidence.
4. **Ask many questions about one state** in one request, and batch many states.
5. **Compare** the same triage on Groq structured output: what an LLM gives you, and what it cannot (calibrated probabilities).
6. **Handle** the failure modes: a replay miss, `429` rate limits, a bad key.

## 📚 Official reading (verified 2026-10-16)

| Page | Read it for |
|---|---|
| [TypeSafe integration (LangChain)](https://docs.langchain.com/oss/python/integrations/providers/typesafe) | `TypeSafeClassifier`, `Noul`/`Choice`/`Score`, the experimental middleware |
| [What is Jev? (LangChain blog)](https://www.langchain.com/blog/building-a-harness-with-jev) | System One models, RLCD, where Jev fits in a harness |
| [Building prod with Jev and LangGraph](https://www.langchain.com/blog/building-prod-with-jev-and-langgraph) | "Prod, not god"; "cheap by default, frontier on exception" |
| [TypeSafe quickstart](https://docs.typesafe.ai/introduction/quickstart) | The `POST /v1/systemone` request and response |
| [TypeSafe models](https://docs.typesafe.ai/models) | Model IDs, price, context length, rate limits |

> **Why this matters at work.** Orbita Cloud's support desk (Module 14) routes about 4,000 tickets a day. Every ticket needs a few *judgments*: which team owns it, how urgent it is, whether it reports a bug, and whether the customer might leave. An LLM can make those calls, but it pays for writing prose it never writes, takes seconds, and gives a different answer on a re-run. A decision model makes only the judgment. It is fast, it costs fractions of a cent, and it returns a probability your code can set a threshold on. The LLM is kept for the tickets the decision model is unsure about.

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain>=1.4.4,<2" "langchain-core>=1.6.9,<2" "langchain-typesafe[experimental]==0.0.1a3" "langchain-groq>=1.1.3,<2"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain", "langchain-core", "langchain-typesafe", "langchain-groq",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Setup 1: Find the repository root (Jupyter starts in modules/15-decision-models-with-jev/).
import importlib.util
import os
import sys
from pathlib import Path

# Optional: set LC_MASTERY_ROOT to your clone if Jupyter starts outside it (no paths are hardcoded).
_override = os.environ.get("LC_MASTERY_ROOT", "").strip()
REPO_ROOT = next(
    (p for p in ([Path(_override).expanduser()] if _override else [Path.cwd(), *Path.cwd().parents])
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone (or set LC_MASTERY_ROOT to it).")

# Setup 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('TYPESAFE_API_KEY', 'GROQ_API_KEY')
LANGSMITH_PROJECT = 'm15-decision-models-vs-llms'
# Setup 3: Load .env, check keys WITHOUT printing them, and build Jev in live or replay mode.
import os

from mastery.config import env_status, is_set, load_env, load_settings
from mastery.decisions import decision_classifier
from mastery.doctor import check_decisions_env

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")
HAVE_TYPESAFE_KEY = is_set("TYPESAFE_API_KEY")
os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)   # m15-<lesson>: one tracing project per lesson
JEV = decision_classifier()                                      # live with a key, replay without (never raises offline)
jev = JEV.classifier

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("project   :", os.environ["LANGSMITH_PROJECT"])
print("fast      :", settings.model_fast, "(Groq)")
print("jev       :", JEV.describe())
for note in check_decisions_env():
    print("⚠️", note)
if JEV.mode == "replay":
    print("\nℹ️ Replay mode: ⚖️ cells answer from recorded Jev responses (marked synthetic until real ones are recorded).")
    print("   They show the API and the decision logic; they are NOT measurements of Jev. Add TYPESAFE_API_KEY to go live.")
if not HAVE_GROQ_KEY:
    print("⚠️ No GROQ_API_KEY: 💸 Groq cells will print ⏭️ Skipped; 🧪 and ⚖️ cells still run.")

**You should now see** your key status (never the key itself) and a `jev :` line. Without `TYPESAFE_API_KEY` it reads `replay · <N> recorded requests in recordings.json (100% synthetic) · no key, no network, $0`, where N is the number of recorded requests across all Module 15 lessons (it grows as lessons are added). With a key it reads `live · jev-1.13.0 at https://api.typesafe.ai`. The ℹ️ note about synthetic answers is there on purpose: keep it in mind for every ⚖️ cell.

In [ ]:
# Setup 4: Silence known, harmless noise so real problems stand out.
import os
import warnings
from importlib.metadata import version

# langsmith warns once when a Client is created without a key; offline cells never send anything.
warnings.filterwarnings("ignore", message="API key must be provided when using hosted LangSmith API")
# Same as Lesson 02.04: older langchain-core 1.6.x Pydantic serializer warning about the injected runtime.
# Only older releases raise it; on current ones this filter is skipped.
if tuple(int("".join(c for c in x if c.isdigit()) or 0) for x in version("langchain-core").split(".")[:3]) < (1, 6, 11):
    warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# Preview models warn once per model build; the lesson says so in Markdown already.
from mastery.models import PreviewModelWarning
warnings.filterwarnings("once", category=PreviewModelWarning)
# langchain-typesafe 0.0.1a3 marks TypeSafeClassifier as beta on every construction; the lesson says so.
warnings.filterwarnings("ignore", message=".*TypeSafeClassifier.*beta")

In [ ]:
def table(rows, columns):
    """Print dict rows as an aligned text table (one comparison cell per Part)."""
    widths = {c: max(len(c), *(len(str(r.get(c, ""))) for r in rows)) for c in columns}
    print("  ".join(c.ljust(widths[c]) for c in columns))
    for r in rows:
        print("  ".join(str(r.get(c, "")).ljust(widths[c]) for c in columns))

import time

from mastery.decisions import answer_rows, decision_cost, provenance
from mastery.decisions.orbita import TEAMS, TICKETS, URGENCY, ticket_state, triage_questions, urgent_question

for t in TICKETS:
    print(f"{t['id']} [{t['tier']:<10}] {t['text']}")

**You should now see** six Orbita Cloud tickets. T1–T4 are clear-cut. **T5** mixes a login problem with a billing problem, and **T6** is a slow-export complaint with a hint of churn. Real queues are full of tickets like T5 and T6, and they are where a decision's *probability* earns its keep.

---
## Part 1 · What a decision model is

A chat LLM **generates text**. Even with structured output, it writes tokens that a parser turns into your schema. A **decision model** never writes. You hand it a **state** (text, JSON or messages) and **named questions** whose possible answers you define up front, and it returns a typed answer for each one, with probabilities.

| | Chat LLM (+ structured output) | Decision model (Jev) |
|---|---|---|
| Output | Generated tokens, parsed into your schema | Typed answers only: no text, so nothing to parse or hallucinate |
| Uncertainty | None, or a number the model *writes* (not calibrated) | A probability for every option, trained to be calibrated (RLCD) |
| Re-run the same input | Can change (sampling) | Designed to return the same answer |
| Speed / cost | Seconds; you pay for output tokens | ~70–500 ms (vendor figures); input tokens only, output free |
| Can it explain, write, plan? | Yes | **No**: it only decides |

TypeSafe calls this a **System One** model, after the "fast, intuitive" half of Kahneman's thinking model. The LLM is System Two: slow and deliberate. The three question types:

| Type | You define | You get back |
|---|---|---|
| `Noul(instructions, criteria=NoulCriteria(true=…, false=…))` | A yes/no question (criteria optional) | `.noul`: the **probability of yes** (0–1) |
| `Choice(instructions, criteria={label: description})` | Named options | `.choice`, `.probabilities` per label, `.confidence` |
| `Score(instructions, criteria=[level0, level1, …])` | An ordered rubric | `.score` (expected level, can be fractional), `.probabilities` per level, `.legend`, `.confidence` |

*Noul* is TypeSafe's name for a "probabilistic boolean". The answer is a probability and not `True`/`False` on purpose: **you** choose the cutoff, based on what a wrong answer costs (Lesson 15.02).

---
## Part 2 · Getting access (and what it costs)

1. **Playground (free):** sign up at [console.typesafe.ai](https://console.typesafe.ai) and try questions by hand in the browser.
2. **API key (paid):** new accounts get **no free credit** (verified 2026-10-16), and the console refuses to create a key while the balance is $0. Add credit under **Billing**, then create a key and put it in `.env` as `TYPESAFE_API_KEY`.
3. **Price:** $0.042 per 1M **input** tokens; output is free. A ticket with four questions is about 300–500 input tokens: roughly $0.00002.
4. **Through a gateway instead:** OpenRouter (`TYPESAFE_BASE_URL=https://openrouter.ai/api`, your OpenRouter key, model `typesafe/jev-1.13`) and Vercel AI Gateway implement the same System One API.
5. **No key at all:** replay mode (this notebook's default without a key).

**Pin the model.** `jev-latest` follows each new release: it is 1.13 today and moves to 1.14 around 2026-10-21. The course pins `jev-1.13.0` (`MASTERY_DECISION_MODEL`), so a decision you tuned does not change under you.

🧪 The cell below runs offline. It prints what the doctor thinks of your Jev settings and what this lesson's calls would cost.

In [ ]:
from mastery.doctor import check_decisions_env
from mastery.pricing import get_price

price = get_price(JEV.model)
print("mode      :", JEV.mode, "| model:", JEV.model)
print("doctor    :", check_decisions_env() or "no problems found")
print(f"price     : ${price.input_per_m} per 1M input tokens, ${price.output_per_m} output (verified {price.verified})")
print(f"this lesson: ~7 requests × ~350 tokens ≈ ${7 * 350 * price.input_per_m / 1e6:.6f}")

**You should now see** `doctor : no problems found`, the price line, and a lesson cost of about `$0.0001`. It costs that little because a decision model bills only the text it reads.

---
## Part 3 · First call: one yes/no question

> ⚖️ **Jev cell.** With `TYPESAFE_API_KEY` it calls TypeSafe (billed at $0.042 per 1M input tokens, a fraction of a cent here). Without a key it **replays** recorded answers: same API, same objects, no network. Until real answers are recorded, replayed answers are *synthetic* (hand-written from the documented schema): learn the API and the logic from them, not Jev's accuracy.

In [ ]:
# ⚖️ One state, one Noul question. The input is a plain dict: {"state": ..., "questions": {name: Question}}.
start = time.perf_counter()
r = jev.invoke({"state": ticket_state(TICKETS[0]), "questions": urgent_question()})
ms = (time.perf_counter() - start) * 1000

p = r.nouls["urgent"].noul          # a float: the probability that the answer is "yes"
print(type(r).__name__, "·", provenance(r), "· model", r.model, f"· {ms:.0f} ms")
print(f"p(urgent) = {p:.2f}  →  {'page the on-call engineer' if p >= 0.8 else 'normal queue'}  (threshold 0.8 is OUR choice)")
print("usage     :", r.usage, f"· cost ${decision_cost(r):.8f}")

**You should now see** `ClassifierResponse · replay (synthetic)` (or `live`), `p(urgent) = 0.99 → page the on-call engineer`, and a cost of a few millionths of a dollar. In replay mode, the milliseconds are the local replay and not Jev. Lesson 15.07 measures real latency.

Three things to notice:

- `r.nouls["urgent"]` has a **probability** and no label. The cutoff (`0.8`) lives in **your** code, where you can test it and change it.
- `r` is a `ClassifierResponse`, and nothing needed parsing. `r.answers` holds every answer by name, and `r.nouls`, `r.choices` and `r.scores` group them by type.
- `provenance(r)` (a course helper) reads the response's `request_id` to tell a live answer from a replayed one. Always know which one you are looking at.

---
## Part 4 · Four questions about one ticket, one request

> ⚖️ **Jev cell.** With `TYPESAFE_API_KEY` it calls TypeSafe (billed at $0.042 per 1M input tokens, a fraction of a cent here). Without a key it **replays** recorded answers: same API, same objects, no network. Until real answers are recorded, replayed answers are *synthetic* (hand-written from the documented schema): learn the API and the logic from them, not Jev's accuracy.

In [ ]:
questions = triage_questions()           # team (Choice), urgency (Score), is_bug and churn_risk (Noul)
for name, q in questions.items():
    print(f"{name:<11} {q.type:<7} {q.instructions}")

r = jev.invoke({"state": ticket_state(TICKETS[4]), "questions": questions})    # T5: the ambiguous one
print("\nprovenance:", provenance(r), "| usage:", r.usage)
table(answer_rows(r), ["question", "type", "answer", "p_yes", "confidence", "distribution"])

team = r.choices["team"]
print(f"\nteam = {team.choice!r}, confidence {team.confidence:.2f}; runner-up probabilities:", team.probabilities)
urgency = r.scores["urgency"]
print(f"urgency = {urgency.score:.2f} on the 0..2 rubric → nearest level: {urgency.legend[round(urgency.score)]!r}")

**You should now see** the four questions, then a table for T5: `team` is `account_access` with `billing` close behind (a confidence around `0.08`), `urgency` is about `1.0`, and `is_bug` sits near `0.5`. This is the ticket a human would also hesitate on, and the numbers say so. An LLM would just pick one.

- The questions in one request are **answered independently, in parallel**, against the same state. Four questions cost about the same as one, because you pay for the state's tokens, not the answers.
- `confidence` is TypeSafe's single-number summary of how peaked the distribution is. Keep the full `probabilities` when the *runner-up* matters. Here it says "also billing".
- The `Score` answer is an **expected value** (`1.0` = mostly "handled today"), not an integer. Use `.probabilities` if you need "P(blocking) > 0.2".
- The state is **JSON** (`customer_tier` + `ticket`), so the tier is part of what Jev reads.

---
## Part 5 · Batch the queue, and route on confidence

> ⚖️ **Jev cell.** With `TYPESAFE_API_KEY` it calls TypeSafe (billed at $0.042 per 1M input tokens, a fraction of a cent here). Without a key it **replays** recorded answers: same API, same objects, no network. Until real answers are recorded, replayed answers are *synthetic* (hand-written from the documented schema): learn the API and the logic from them, not Jev's accuracy.

In [ ]:
# ⚖️ TypeSafeClassifier is a Runnable: .batch() sends the requests concurrently.
inputs = [{"state": ticket_state(t), "questions": triage_questions()} for t in TICKETS]
start = time.perf_counter()
results = jev.batch(inputs)
seconds = time.perf_counter() - start

ROUTE_AT = 0.6    # team confidence needed to auto-route; below it, a stronger model (or a human) decides
rows, total_cost = [], 0.0
for t, res in zip(TICKETS, results):
    team, urgency = res.choices["team"], res.scores["urgency"]
    route = f"→ {team.choice}" if team.confidence >= ROUTE_AT else "→ ESCALATE (LLM)"
    rows.append({"id": t["id"], "team": team.choice, "conf": f"{team.confidence:.2f}", "urgency": f"{urgency.score:.2f}",
                 "bug": f"{res.nouls['is_bug'].noul:.2f}", "churn": f"{res.nouls['churn_risk'].noul:.2f}", "route": route})
    total_cost += decision_cost(res) or 0.0
table(rows, ["id", "team", "conf", "urgency", "bug", "churn", "route"])
auto = sum(not r["route"].endswith("(LLM)") for r in rows)
print(f"\n{auto}/{len(rows)} auto-routed · {len(rows) - auto} escalated · {provenance(results[0])} · "
      f"{seconds:.2f} s · total ${total_cost:.6f}")

**You should now see** T1–T4 routed to `technical`, `billing`, `account_access` and `feature_request` with high confidence, and **T5 and T6 escalated**. The summary line reads `4/6 auto-routed · 2 escalated` and a total under $0.0001. T6 also has a high `churn` probability: that is a signal for an account manager, which an LLM-only router would not give you as a number.

This is the module's core pattern in eight lines: **cheap by default, frontier on exception.** The decision model handles the bounded choices it is sure about, and only the uncertain ones pay for an LLM. `ROUTE_AT = 0.6` is a guess for now; Lesson 15.02 picks it from labeled data and the cost of each mistake.

---
## Part 6 · The same triage with an LLM (Groq structured output)

For comparison, the same four judgments come from Groq's fast model through `structured_model` (strict JSON schema, the course's fix for Groq `tool_use_failed`). Watch three things: latency, whether the labels agree, and what is **missing** (probabilities).

In [ ]:
# 💸 Six Groq calls with the fast model (about 2–3K tokens of your free-tier quota).
from typing import Literal

from pydantic import BaseModel, Field

from mastery.models import get_model, structured_model


class Triage(BaseModel):
    """Triage decision for one support ticket."""
    team: Literal["billing", "account_access", "technical", "feature_request"]
    urgency: int = Field(ge=0, le=2, description="0 = can wait, 1 = today, 2 = blocking right now")
    is_bug: bool = Field(description="Broken product behaviour (not a question, request or billing matter)")
    churn_risk: bool = Field(description="The customer signals they might cancel or not renew")


if not HAVE_GROQ_KEY:
    print("⏭️ Skipped: GROQ_API_KEY is not set (see the setup cell).")
else:
    llm_triage = structured_model(get_model("fast", temperature=0), Triage)
    guide = "Teams: " + "; ".join(f"{k}: {v}" for k, v in TEAMS.items())
    rows = []
    for t, res in zip(TICKETS, results):
        start = time.perf_counter()
        out = llm_triage.invoke(f"{guide}\n\nCustomer tier: {t['tier']}\nTicket: {t['text']}")
        rows.append({"id": t["id"], "llm_team": out.team, "jev_team": res.choices["team"].choice,
                     "agree": "✅" if out.team == res.choices["team"].choice else "≠",
                     "llm_urg": out.urgency, "llm_bug": out.is_bug, "llm_churn": out.churn_risk,
                     "llm_s": f"{time.perf_counter() - start:.2f}"})
    table(rows, ["id", "llm_team", "jev_team", "agree", "llm_urg", "llm_bug", "llm_churn", "llm_s"])
    print("\nThe LLM gives labels and booleans, no probabilities: T5 and T6 look exactly as certain as T1.")

**You should now see** (with a Groq key) a table where the LLM agrees with Jev on T1–T4, picks a single team for T5 with no hint that it was a close call, and answers each call in roughly 0.3–1.5 s. Without a key the cell prints `⏭️ Skipped`.

> **"Can't I just ask the LLM how confident it is?"** You can add a `confidence: float` field, but that number is *generated text*. It is not trained to match how often the model is right, and it often reads `0.9` on the cases the model gets wrong. Lesson 15.02 measures this (reliability diagrams) for both models on labeled tickets. Jev's calibration is a vendor claim too, and you will check it rather than take it on trust.

---
## Part 7 · When things go wrong

In [ ]:
# 🧪 Offline: the three failures you will meet first, all caught.
import httpx2
from langchain_typesafe import TypeSafeClassifier
from langchain_typesafe.client import TypeSafeAuthenticationError, TypeSafeRateLimitError

from mastery.decisions import ReplayMiss, with_rate_limit_retry

# 1) Replay mode only knows the lesson's own requests. New text = a clear ReplayMiss.
try:
    new = jev.invoke({"state": "Our webhook signatures stopped validating after your update.", "questions": urgent_question()})
    print("live answer for new text: p(urgent) =", round(new.nouls["urgent"].noul, 2))
except ReplayMiss as exc:
    print("⛔ ReplayMiss (expected in replay mode):", str(exc)[:95], "…")

# 2) 429: TypeSafe limits are per organization (tokens/s and requests/s). Simulated here with a fake server.
calls = []
def busy_then_ok(request):
    calls.append(1)
    if len(calls) == 1:
        return httpx2.Response(429, headers={"retry-after": "0"}, json={"error": "rate limited"})
    return httpx2.Response(200, json={"model": "jev-1.13.0", "answers": {"urgent": {"type": "noul", "noul": 0.97}},
                                      "usage": {"input_tokens": 40}})
fake = TypeSafeClassifier(api_key="fake", client=httpx2.Client(transport=httpx2.MockTransport(busy_then_ok)))
try:
    fake.invoke({"state": "x", "questions": urgent_question()})
except TypeSafeRateLimitError as exc:
    print("⛔ TypeSafeRateLimitError without retry (expected):", exc.status, "retry_after_ms =", exc.retry_after_ms)
calls.clear()                                   # fresh fake server: first call 429 again, then 200
ok = with_rate_limit_retry(fake).invoke({"state": "x", "questions": urgent_question()})
print("with_rate_limit_retry:", len(calls), "attempts → p(urgent) =", ok.nouls["urgent"].noul)

# 3) A wrong or unfunded key: 401/403, never retried (it would fail the same way).
deny = TypeSafeClassifier(api_key="wrong", client=httpx2.Client(transport=httpx2.MockTransport(
    lambda request: httpx2.Response(401, json={"error": "invalid api key"}))))
try:
    with_rate_limit_retry(deny).invoke({"state": "x", "questions": urgent_question()})
except TypeSafeAuthenticationError as exc:
    print("⛔ TypeSafeAuthenticationError (expected):", exc.status, "→ check TYPESAFE_API_KEY and your credit balance")

**You should now see** three `⛔ … (expected)` lines, plus `with_rate_limit_retry: 2 attempts → p(urgent) = 0.97`: the first attempt got a 429, and the retry succeeded. With a live key, line 1 prints a real answer for the new text instead of the `ReplayMiss`.

- `langchain-typesafe` 0.0.1a3 does **not** retry by itself. TypeSafe's own SDKs do, which is why `with_rate_limit_retry` exists. It retries **only** `429`.
- Every error subclasses LangChain's model errors (`TypeSafeRateLimitError` is a `ModelRateLimitError`), so the fallback tools from Module 12 recognize them.
- Every fake call here goes through `httpx2.MockTransport`. That is the same mechanism replay mode uses, and it is how you unit-test code that calls Jev.

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `API keys cannot be created … no available credits` (console) | No free tier for new accounts | Add credit in Billing, or keep using replay mode / the Playground |
| `ValueError: MASTERY_DECISIONS_MODE=live needs TYPESAFE_API_KEY` | Forced live mode without a key | Set the key, or `MASTERY_DECISIONS_MODE=auto` |
| `ReplayMiss` | New text or edited questions in replay mode | Expected; go live, or use the lesson's inputs |
| `404` on every call | `TYPESAFE_BASE_URL` ends in `/v1` | Use the root (`https://openrouter.ai/api`); the client adds `/v1/systemone` |
| `TypeSafeRateLimitError` (429) | Org limits in tokens/s or requests/s | `with_rate_limit_retry`, smaller batches |
| `LangChainBetaWarning` | The package is an alpha | Expected; the setup cell silences it |
| Answers changed after an upgrade | `jev-latest` moved to a new release | Pin `jev-1.13.0`; re-evaluate before switching (15.02) |

## ⚠️ Production notes

> ⚠️ **Production note: a probability is not a decision.** (Parts 3, 5)
> Thresholds such as `0.8` and `0.6` are business rules. Keep them in config, choose them from labeled data and the cost of each kind of mistake (Lesson 15.02), and log the probability with every routed ticket, so you can re-tune later without re-running Jev.

> ⚠️ **Production note: the alpha will move.** (All parts)
> `langchain-typesafe` is `0.0.1a3`, and the middleware is experimental. Pin the version, wrap it (as `mastery.decisions` does), and keep a test that fails loudly when its import surface or response fields change.

## 🏋️ Exercises

1. **(Easy)** Change Part 3's threshold to `0.995`. What happens to T1, and what does that tell you about where the threshold should come from?
2. **(Medium)** Add a fifth question, `contains_pii` (`Noul`), to `triage_questions()` in your own copy (not in `src/`), and ask it about all six tickets. Why does replay mode refuse it, and what does that show about how replay finds answers?
3. **(Medium)** In Part 5, sweep `ROUTE_AT` over `0.3, 0.5, 0.6, 0.8, 0.95` and print how many tickets escalate for each. Which value would you ship before you have labeled data?
4. **(Stretch, needs a key)** Run Part 4 five times live, and the Part 6 LLM five times at `temperature=0.7`. Which one changes its answer for T5?

<details>
<summary><b>Show solutions</b></summary>

**1.** `0.99 < 0.995`, so T1 drops to "normal queue" even though it is a live outage. A threshold that is too strict throws away a clearly correct decision. Pick it from what a missed page costs compared with a false page, not from how sure you would like the model to be.

**2.** `q = triage_questions(); q["contains_pii"] = Noul(instructions="Does the ticket contain personal data (names, emails, card numbers)?")`. Replay looks answers up by a hash of **state + questions**, so a changed question set is a new request and gives a `ReplayMiss`. Live, you pay roughly the same as before: the state dominates the token count.

**3.** `for th in (...): print(th, sum(res.choices["team"].confidence < th for res in results))`. With these answers, `0.6` escalates 2 of 6 tickets and `0.95` escalates most of them. Before you have labels, pick a conservative value and **log** every probability. Lesson 15.02 turns those logs into a threshold.

**4.** Jev is designed to give the same answer for the same input, so expect identical numbers. The LLM at `0.7` will sometimes flip T5 between `account_access` and `billing`. That instability is a big part of why decision models make good judges (Lesson 15.06).
</details>

## 🔎 Docs drill (5 min)

Which HTTP header makes Jev accept up to 262k tokens per request, and what are the default limits without it?

Try it before you open the answer: use the official docs, the API reference, or the installed source (`??` in Jupyter).

<details>
<summary>Path to the answer</summary>

1. Open [TypeSafe models](https://docs.typesafe.ai/models) and find **Long context**.
2. The header is **`TypeSafe-Danger-Zone-Long-Context: true`**, sent per request.
3. Without it, the limits are **64k tokens per request and 32k per question**. With it, both are 262k, at the same price.

</details>

## ✅ Checkpoint

- [ ] I can explain in two sentences how a decision model differs from an LLM with structured output.
- [ ] I know what Jev access costs, that there is no free tier, and what replay mode does (and does not) prove.
- [ ] I asked `Noul`, `Choice` and `Score` questions and read `.nouls`, `.choices`, `.scores`, `probabilities` and `confidence`.
- [ ] I routed six tickets on a confidence threshold and saw which ones escalate.
- [ ] Restart & Run All passes with no keys (⚖️ replays, 💸 skips); outputs cleared before saving.

## 🔎 Research checkpoint

On [TypeSafe models](https://docs.typesafe.ai/models), check which release `jev-latest` points to today and whether the price changed. Log both in [`docs/research-log.md`](../../docs/research-log.md) with the date.

## Summary

- A decision model answers **typed questions** about a **state** with **probabilities**, and writes no text. Jev is TypeSafe's first one.
- `TypeSafeClassifier.invoke({"state": ..., "questions": {...}})` returns a `ClassifierResponse`. Read it with `.nouls[...]`, `.choices[...]` and `.scores[...]`.
- Many questions about one state cost about the same as one. You pay for input tokens only.
- **Cheap by default, frontier on exception:** route confident answers in code, and escalate uncertain ones to an LLM or a human.
- The course pins `langchain-typesafe==0.0.1a3` and `jev-1.13.0`. Without a key, replay mode keeps every cell running.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 15-02-calibration-and-thresholds.ipynb** → [`15-02-calibration-and-thresholds.ipynb`](15-02-calibration-and-thresholds.ipynb).

[Module 15 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)